# LINet3 t_max Discovery on SUN RGB-D

**Find the natural training duration (t_max) before launching hyperparameter sweeps.**

---

## Methodology:

1. **Unbounded Baseline Run** — vanilla config, ReduceLROnPlateau, MAX_EPOCHS=200
2. **Watch the inflection point** — where val_loss plateaus while train_loss keeps dropping
3. **That epoch = your t_max** for Cosine Annealing in HPO sweeps

The Plateau scheduler is *reactive* — it holds LR steady until val_loss stalls,
then steps it down. Early stopping kills the run when no further progress is possible.

---

## Checklist:

- [ ] **Enable A100 GPU**
- [ ] **Upload dataset:** `MyDrive/datasets/sunrgbd_19_traintest.tar.gz`


## 1. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\nA100 GPU detected - optimal for training")
    elif 'V100' in gpu_name:
        print("\nV100 GPU detected - good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\nT4 GPU detected - will be slower, consider upgrading to A100")
    else:
        print(f"\nGPU: {gpu_name}")
else:
    print("\nNO GPU DETECTED!")
    print("Enable GPU: Runtime -> Change runtime type -> Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

In [ ]:
# Detailed GPU info
!nvidia-smi

Fri Feb 27 01:02:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   40C    P0             54W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

## 3. Clone Repository to Local Disk (Fast I/O)

**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

**Default:** Clone from GitHub (recommended - always gets latest code)

In [ ]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

os.chdir('/content')

if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"Repo already exists: {LOCAL_REPO_PATH}")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
else:
    if Path(LOCAL_REPO_PATH).exists():
        !rm -rf {LOCAL_REPO_PATH}
    print(f"Cloning from {GITHUB_REPO}...")
    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository")
    os.chdir(LOCAL_REPO_PATH)

print(f"\nWorking directory: {os.getcwd()}")
!ls -la {LOCAL_REPO_PATH}
print("\n" + "=" * 60)

## 4. Install Dependencies

In [ ]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn kornia

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import kornia

print("All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   kornia: {kornia.__version__}")

## 5. Copy SUN RGB-D Dataset to Local Disk

**Performance Note:** Local disk I/O is ~10-20x faster than Drive!

**Dataset:** SUN RGB-D 15-category preprocessed (train + test splits, RGB + Depth)

In [ ]:
from pathlib import Path
import os

# Paths
DRIVE_DATASET_TAR = "/content/drive/MyDrive/datasets/sunrgbd_19_traintest.tar.gz"
LOCAL_DATASET_PATH = "/dev/shm/sunrgbd_19_traintest"

print("=" * 60)
print("SUN RGB-D 19-CATEGORY DATASET SETUP")
print("=" * 60)

if Path(LOCAL_DATASET_PATH).exists():
    print(f"Already on local disk: {LOCAL_DATASET_PATH}")
    train_count = len(list(Path(f"{LOCAL_DATASET_PATH}/train/rgb").glob("*.png")))
    print(f"   Train samples: {train_count}")
elif Path(DRIVE_DATASET_TAR).exists():
    print(f"Found on Drive: {DRIVE_DATASET_TAR}")
    tar_name = Path(DRIVE_DATASET_TAR).name
    local_tar = f"/dev/shm/{tar_name}"
    !rsync -ah --info=progress2 {DRIVE_DATASET_TAR} {local_tar}
    print(f"\nExtracting...")
    !tar -xzf {local_tar} -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"
    !rm {local_tar}
    train_count = len(list(Path(f"{LOCAL_DATASET_PATH}/train/rgb").glob("*.png")))
    print(f"Extracted. Train samples: {train_count}")
else:
    raise FileNotFoundError(f"Dataset not found at {DRIVE_DATASET_TAR}")

print(f"\nDataset ready at: {LOCAL_DATASET_PATH}")


## 6. Setup Python Path & Import LINet3

In [ ]:
import sys
import os

# Remove cached modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

# Add project to Python path
project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Verify project structure
print("Project structure:")
!ls -la {project_root}/src/models/

# Import LiNet and dataloaders
print("\nImporting LiNet, dataloaders, and visualization tools...")
from src.models.linear_integration.li_net3 import li_resnet18
from src.data_utils.sunrgbd_dataset import get_sunrgbd_dataloaders
from src.training.augmentation_config import AugmentationConfig

# Import visualization suite
from src.utils.visualization import (
    FeatureMapVisualizer,
    StreamContributionVisualizer,
    StreamGradCAM,
    IntegrationWeightVisualizer,
    find_misclassified,
    compare_samples,
    StreamRedundancyAnalyzer,
    PerClassDominanceAnalyzer,
    ActivationDivergenceAnalyzer,
    IntegrationWeightEvolutionVisualizer,
    reset_bn_stats,
)

print("All imports successful!")

In [ ]:
# Set random seed for reproducibility
from src.utils.seed import set_seed

SEED = 42
DETERMINISTIC = False  # False = faster, True = fully reproducible

set_seed(SEED, deterministic=DETERMINISTIC)

print(f"Seed: {SEED}, Deterministic: {DETERMINISTIC}")

## 7. Configuration

All hyperparameters and settings in one place. Modify these before running.

In [ ]:
from src.training.augmentation_config import AugmentationConfig
from sklearn.model_selection import train_test_split

# =============================================================================
# t_max DISCOVERY CONFIGURATION
# =============================================================================
# Vanilla defaults — minimal regularization so the model reveals its
# natural training duration. ReduceLROnPlateau watches val_loss and
# steps LR down when progress stalls. Early stopping kills the run
# when further training would only overfit.
# =============================================================================

MAX_EPOCHS = 200  # Ceiling — early stopping will kick in well before this

# ======================== DATASET ========================
DATASET_CONFIG = {
    'data_root': LOCAL_DATASET_PATH,
    'batch_size': 64,
    'num_workers': 8,
    'num_classes': 19,
    'seed': SEED,
}

# Light augmentation — just enough to prevent trivial overfitting
AUGMENTATION_CONFIG = AugmentationConfig(
    rgb_aug_prob=1.0,
    rgb_aug_mag=1.0,
    depth_aug_prob=1.0,
    depth_aug_mag=1.0,
)

# ======================== MODEL ========================
MODEL_CONFIG = {
    'architecture': 'resnet18',
    'num_classes': 19,
    'stream_input_channels': [3, 1],
    'width_multiplier': 0.75,
    'dropout_p': 0.3,  # Light dropout
    'device': 'cuda',
    'use_amp': True,
}

STREAM_LABELS = {0: 'RGB', 1: 'Depth'}

# ======================== OPTIMIZER ========================
# Standard literature defaults — nothing fancy
STREAM_SPECIFIC_CONFIG = {
    'stream_lrs': [1e-4, 1e-4],
    'shared_lr': 5e-5,
    'stream_weight_decays': [1e-4, 1e-4],
    'integration_weight_decay': 1e-4,
}

# ======================== SCHEDULER ========================
# ReduceLROnPlateau: reactive, watches val_loss
SCHEDULER_CONFIG = {
    'scheduler_type': 'plateau',
    'mode': 'min',           # Watch val_loss (lower is better)
    'patience': 5,           # Wait 5 epochs before reducing LR
    'factor': 0.5,           # Halve LR on plateau
    'min_lr': 1e-7,          # Floor — stop reducing below this
    'warmup_epochs': 5,
    'warmup_start_factor': 0.1,
}

# ======================== TRAINING ========================
TRAIN_CONFIG = {
    'epochs': MAX_EPOCHS,
    'grad_clip_norm': 1.0,
    'early_stopping': True,
    'patience': 15,          # Stop if val_loss doesn't improve for 15 epochs
    'monitor': 'val_loss',
    'restore_best_weights': True,
    'stream_monitoring': True,
    'modality_dropout': True,
    'modality_dropout_start': 0,
    'modality_dropout_ramp': 10,
    'modality_dropout_rate': 0.1,
    'label_smoothing': 0.05,  # Light
    'gradient_monitoring': True,
    'gradient_log_freq': 0,
    'track_integration_weights': True,
    'integration_snapshot_freq': 10,
}

print('t_max Discovery Config:')
print(f'  MAX_EPOCHS: {MAX_EPOCHS}')
print(f'  Scheduler: {SCHEDULER_CONFIG["scheduler_type"]} (patience={SCHEDULER_CONFIG["patience"]}, factor={SCHEDULER_CONFIG["factor"]})')
print(f'  Early stopping: patience={TRAIN_CONFIG["patience"]} on {TRAIN_CONFIG["monitor"]}')
print(f'  LR floor: {SCHEDULER_CONFIG["min_lr"]}')
print(f'  Dropout: {MODEL_CONFIG["dropout_p"]} (light)')
print(f'  Label smoothing: {TRAIN_CONFIG["label_smoothing"]} (light)')


## 8. Load Dataset

In [ ]:
from pathlib import Path

print("=" * 60)
print("DATASET STRUCTURE VERIFICATION")
print("=" * 60)

dataset_root = Path(LOCAL_DATASET_PATH)

for split in ['train', 'test']:
    split_dir = dataset_root / split
    if split_dir.exists():
        print(f"  {split}/")
        for modality in ['rgb', 'depth']:
            mod_dir = split_dir / modality
            if mod_dir.exists():
                print(f"    {modality}/ - {len(list(mod_dir.glob('*.png')))} images")

class_names_file = dataset_root / 'class_names.txt'
if class_names_file.exists():
    with open(class_names_file, 'r') as f:
        class_names = [line.strip().split(': ', 1)[-1] if ': ' in line.strip() else line.strip() for line in f if line.strip()]
    print(f"\nClasses ({len(class_names)}):")
    for i, name in enumerate(class_names):
        print(f"  {i}: {name}")


In [ ]:
import torch
import numpy as np
import random
from collections import Counter

print("=" * 60)
print("LOADING SUN RGB-D 19-CATEGORY (80/20 TRAIN/VAL SPLIT)")
print("=" * 60)

# Two dataset instances from train/ directory
train_dataset = SUNRGBDDataset(
    data_root=DATASET_CONFIG['data_root'],
    split='train',
    normalize=True,
    **AUGMENTATION_CONFIG.to_dict(),
)
val_dataset = SUNRGBDDataset(
    data_root=DATASET_CONFIG['data_root'],
    split='train',
    normalize=True,
)
val_dataset.split = 'val'  # Disable augmentation

# Locked 80/20 stratified split
all_labels = train_dataset.labels
train_indices, val_indices = train_test_split(
    list(range(len(all_labels))),
    test_size=0.2,
    random_state=SEED,
    stratify=all_labels,
)

train_subset = torch.utils.data.Subset(train_dataset, train_indices)
val_subset = torch.utils.data.Subset(val_dataset, val_indices)

# Stratified sampling
subset_labels = [all_labels[i] for i in train_indices]
label_counts = Counter(subset_labels)
num_samples = len(subset_labels)
class_weights = {label: num_samples / count for label, count in label_counts.items()}
sample_weights = torch.tensor(
    [class_weights[label] for label in subset_labels], dtype=torch.float32
)
g = torch.Generator().manual_seed(SEED)
train_sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights, num_samples=num_samples, replacement=True, generator=g,
)

def worker_init_fn(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

train_loader = torch.utils.data.DataLoader(
    train_subset, batch_size=DATASET_CONFIG['batch_size'],
    shuffle=False, sampler=train_sampler,
    num_workers=DATASET_CONFIG['num_workers'], prefetch_factor=4,
    persistent_workers=True, pin_memory=True, worker_init_fn=worker_init_fn,
)
val_loader = torch.utils.data.DataLoader(
    val_subset, batch_size=DATASET_CONFIG['batch_size'],
    shuffle=False,
    num_workers=DATASET_CONFIG['num_workers'], prefetch_factor=2,
    persistent_workers=False, pin_memory=True, worker_init_fn=worker_init_fn,
)

# Also load test set for final evaluation
test_dataset = SUNRGBDDataset(
    data_root=DATASET_CONFIG['data_root'],
    split='test',
    normalize=True,
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=DATASET_CONFIG['batch_size'],
    shuffle=False,
    num_workers=DATASET_CONFIG['num_workers'], prefetch_factor=2,
    pin_memory=True, worker_init_fn=worker_init_fn,
)

print(f"  Train: {len(train_subset)} samples ({len(train_loader)} batches)")
print(f"  Val:   {len(val_subset)} samples ({len(val_loader)} batches)")
print(f"  Test:  {len(test_dataset)} samples ({len(test_loader)} batches)")

rgb_batch, depth_batch, label_batch = next(iter(train_loader))
print(f"\nBatch shapes: RGB={rgb_batch.shape}, Depth={depth_batch.shape}, Labels={label_batch.shape}")


## 9. Create Model

In [ ]:
from src.models.linear_integration.li_net3 import li_resnet18

print("=" * 60)
print("MODEL CREATION")
print("=" * 60)

model = li_resnet18(
    num_classes=MODEL_CONFIG['num_classes'],
    stream_input_channels=MODEL_CONFIG['stream_input_channels'],
    width_multiplier=MODEL_CONFIG['width_multiplier'],
    dropout_p=MODEL_CONFIG['dropout_p'],
    device=MODEL_CONFIG['device'],
    use_amp=MODEL_CONFIG['use_amp'],
)

total_params = sum(p.numel() for p in model.parameters())

print(f"\nLINet3-{MODEL_CONFIG['architecture'].upper()} created")
print(f"  Total parameters: {total_params:,}")
print(f"  Width multiplier: {MODEL_CONFIG['width_multiplier']}")
print(f"  Dropout: {MODEL_CONFIG['dropout_p']}")
print(f"  Streams: {STREAM_LABELS}")


## 10. Compile Model (Optimizer + Scheduler)

In [ ]:
import os
from datetime import datetime
from pathlib import Path

# Create checkpoint directory on Google Drive (persistent storage)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_dir = f"/content/drive/MyDrive/linet_checkpoints/run_{timestamp}"

Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)

print(f"Checkpoint directory: {checkpoint_dir}")

In [ ]:
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler

print("=" * 60)
print("MODEL COMPILATION (ReduceLROnPlateau)")
print("=" * 60)

TRAIN_CONFIG['save_path'] = f"{checkpoint_dir}/best_model.pt"
TRAIN_CONFIG['integration_snapshot_path'] = f"{checkpoint_dir}/integration_snapshots"

# Create optimizer
optimizer = create_stream_optimizer(
    model,
    optimizer_type='adamw',
    stream_lrs=STREAM_SPECIFIC_CONFIG['stream_lrs'],
    stream_weight_decays=STREAM_SPECIFIC_CONFIG['stream_weight_decays'],
    shared_lr=STREAM_SPECIFIC_CONFIG['shared_lr'],
    integration_weight_decay=STREAM_SPECIFIC_CONFIG['integration_weight_decay'],
)

print(f"Optimizer: {optimizer.__class__.__name__}")
for i, group in enumerate(optimizer.param_groups):
    num_params = sum(p.numel() for p in group['params'])
    print(f"  Group {i+1}: lr={group['lr']:.2e}, wd={group['weight_decay']:.2e}, params={num_params:,}")

# Create ReduceLROnPlateau scheduler
scheduler = setup_scheduler(
    optimizer,
    scheduler_type=SCHEDULER_CONFIG['scheduler_type'],
    mode=SCHEDULER_CONFIG['mode'],
    patience=SCHEDULER_CONFIG['patience'],
    factor=SCHEDULER_CONFIG['factor'],
    min_lr=SCHEDULER_CONFIG['min_lr'],
    warmup_epochs=SCHEDULER_CONFIG['warmup_epochs'],
    warmup_start_factor=SCHEDULER_CONFIG['warmup_start_factor'],
)

model.compile(
    optimizer=optimizer,
    scheduler=scheduler,
    loss='cross_entropy',
    label_smoothing=TRAIN_CONFIG['label_smoothing'],
    gpu_augmentation=False,
    **AUGMENTATION_CONFIG.to_dict(),
)

print(f"\nScheduler: ReduceLROnPlateau (patience={SCHEDULER_CONFIG['patience']}, factor={SCHEDULER_CONFIG['factor']}, min_lr={SCHEDULER_CONFIG['min_lr']})")
print("Model compiled!")


## 11. Train with Full Diagnostics

All diagnostics enabled: gradient health monitoring, per-stream training loss decomposition, integration weight norm tracking + periodic full snapshots, stream-specific accuracy monitoring.

In [ ]:
import warnings
import os

warnings.filterwarnings(
    'ignore',
    message='The epoch parameter in `scheduler.step\\(\\)` was not necessary',
    category=UserWarning
)

os.makedirs(TRAIN_CONFIG['integration_snapshot_path'], exist_ok=True)

print("=" * 60)
print("t_max DISCOVERY RUN (ReduceLROnPlateau + Early Stopping)")
print("=" * 60)
print(f"  MAX_EPOCHS: {TRAIN_CONFIG['epochs']}")
print(f"  Early stopping: patience={TRAIN_CONFIG['patience']} on {TRAIN_CONFIG['monitor']}")
print(f"  Scheduler: plateau (patience={SCHEDULER_CONFIG['patience']}, factor={SCHEDULER_CONFIG['factor']})")
print(f"  LR floor: {SCHEDULER_CONFIG['min_lr']}")
print("\n  Watch for the inflection point:")
print("  -> train_loss keeps dropping, val_loss plateaus/climbs = STOP")
print("  -> That epoch number = your t_max for Cosine Annealing")
print("=" * 60 + "\n")

history = model.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=TRAIN_CONFIG['epochs'],
    verbose=True,
    save_path=TRAIN_CONFIG['save_path'],
    early_stopping=TRAIN_CONFIG['early_stopping'],
    patience=TRAIN_CONFIG['patience'],
    monitor=TRAIN_CONFIG['monitor'],
    restore_best_weights=TRAIN_CONFIG['restore_best_weights'],
    grad_clip_norm=TRAIN_CONFIG['grad_clip_norm'],
    stream_monitoring=TRAIN_CONFIG['stream_monitoring'],
    modality_dropout=TRAIN_CONFIG['modality_dropout'],
    modality_dropout_start=TRAIN_CONFIG['modality_dropout_start'],
    modality_dropout_ramp=TRAIN_CONFIG['modality_dropout_ramp'],
    modality_dropout_rate=TRAIN_CONFIG['modality_dropout_rate'],
    gradient_monitoring=TRAIN_CONFIG['gradient_monitoring'],
    gradient_log_freq=TRAIN_CONFIG['gradient_log_freq'],
    track_integration_weights=TRAIN_CONFIG['track_integration_weights'],
    integration_snapshot_path=TRAIN_CONFIG['integration_snapshot_path'],
    integration_snapshot_freq=TRAIN_CONFIG['integration_snapshot_freq'],
)

# === t_max DISCOVERY RESULT ===
total_epochs = len(history['train_loss'])
best_val_epoch = history['val_loss'].index(min(history['val_loss'])) + 1

print("\n" + "=" * 60)
print("t_max DISCOVERY RESULT")
print("=" * 60)
print(f"  Total epochs trained: {total_epochs}")
print(f"  Best val_loss at epoch: {best_val_epoch}")
print(f"  Best val_loss: {min(history['val_loss']):.4f}")
print(f"  Best val_acc:  {max(history['val_accuracy'])*100:.2f}%")
print(f"\n  >>> Recommended t_max for Cosine Annealing: ~{best_val_epoch} <<<")
print("=" * 60)


## 12. Single-Stream Robustness Evaluation

How much does the model degrade when a stream is missing? Tests full model, RGB-only, and Depth-only.

In [ ]:
print("\n" + "=" * 60)
print("SINGLE-STREAM ROBUSTNESS EVALUATION (TEST SET)")
print("=" * 60)
print("\nTesting model performance with missing streams...\n")

# Evaluate with all streams (normal)
print("[1/3] Evaluating with BOTH streams (normal):")
results_both = model.evaluate(test_loader, stream_monitoring=True)
print(f"      Accuracy: {results_both['accuracy']*100:.2f}%")

# Evaluate with RGB only (Depth blanked)
print("\n[2/3] Evaluating with RGB ONLY (Depth blanked):")
results_rgb_only = model.evaluate(test_loader, stream_monitoring=True, blanked_streams={1})
print(f"      Accuracy: {results_rgb_only['accuracy']*100:.2f}%")

# Evaluate with Depth only (RGB blanked)
print("\n[3/3] Evaluating with DEPTH ONLY (RGB blanked):")
results_depth_only = model.evaluate(test_loader, stream_monitoring=True, blanked_streams={0})
print(f"      Accuracy: {results_depth_only['accuracy']*100:.2f}%")

print("\n" + "=" * 60)
print("ROBUSTNESS SUMMARY")
print("=" * 60)
print(f"\n  Both streams:  {results_both['accuracy']*100:.2f}%")
print(f"  RGB only:      {results_rgb_only['accuracy']*100:.2f}% (Depth missing)")
print(f"  Depth only:    {results_depth_only['accuracy']*100:.2f}% (RGB missing)")

rgb_degradation = (results_both['accuracy'] - results_rgb_only['accuracy']) * 100
depth_degradation = (results_both['accuracy'] - results_depth_only['accuracy']) * 100

print(f"\n  Degradation when Depth missing: {rgb_degradation:+.2f}%")
print(f"  Degradation when RGB missing:   {depth_degradation:+.2f}%")
print("\n" + "=" * 60)

## 13. Test Set Evaluation + Pathway Analysis

In [ ]:
print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

results = model.evaluate(data_loader=test_loader, stream_monitoring=True)

print(f"\nTest Results:")
print(f"  Loss: {results['loss']:.4f}")
print(f"  Overall Accuracy: {results['accuracy']*100:.2f}%")

print(f"\nStream-Specific Performance:")
for i in range(len(MODEL_CONFIG['stream_input_channels'])):
    other = (i + 1) % 2
    solo_acc = results[f'stream_{other}_blanked_acc']
    print(f"  {STREAM_LABELS[i]} Solo Accuracy: {solo_acc*100:.2f}%")
    print(f"  {STREAM_LABELS[i]} Contribution: {results[f'stream_{i}_contribution']*100:+.2f}%")

print(f"\n{'='*60}")
print("PATHWAY ANALYSIS")
print(f"{'='*60}")

pathway_analysis = model.analyze_pathways(data_loader=test_loader)

print(f"\nSamples analyzed: {pathway_analysis['samples_analyzed']}")

print("\nAccuracy:")
print(f"  Full model:      {pathway_analysis['accuracy']['full_model']*100:.2f}%")
for i in range(len(MODEL_CONFIG['stream_input_channels'])):
    acc = pathway_analysis['accuracy'][f'stream{i}_only']
    contrib = pathway_analysis['accuracy'][f'stream{i}_contribution']
    print(f"  {STREAM_LABELS[i]} only:       {acc*100:.2f}%  (contribution ratio: {contrib:.3f})")

print("\nFeature Norms (mean +/- std):")
for i in range(len(MODEL_CONFIG['stream_input_channels'])):
    mean = pathway_analysis['feature_norms'][f'stream{i}_mean']
    std = pathway_analysis['feature_norms'][f'stream{i}_std']
    print(f"  {STREAM_LABELS[i]}:        {mean:.4f} +/- {std:.4f}")
int_mean = pathway_analysis['feature_norms']['integrated_mean']
int_std = pathway_analysis['feature_norms']['integrated_std']
print(f"  Integrated:  {int_mean:.4f} +/- {int_std:.4f}")

print(f"\n{'='*60}")
print("TRAINING SUMMARY")
print(f"{'='*60}")
print(f"  Total epochs:       {len(history['train_loss'])}")
print(f"  Best val_loss epoch: {history['val_loss'].index(min(history['val_loss'])) + 1}")
print(f"  Final train acc:    {history['train_accuracy'][-1]*100:.2f}%")
print(f"  Best val acc:       {max(history['val_accuracy'])*100:.2f}%")
print(f"  Test accuracy:      {results['accuracy']*100:.2f}%")


## 14. Training Curves + Gradient Health + Stream Loss Decomposition

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('t_max Discovery — Training Diagnostics', fontsize=16, fontweight='bold', y=1.02)

stream_train_colors = ['skyblue', 'lightcoral', 'gold', 'lightgreen', 'plum']
stream_val_colors = ['blue', 'red', 'orange', 'green', 'purple']
lr_colors = ['blue', 'red', 'orange', 'purple', 'brown']
n_streams = len(MODEL_CONFIG['stream_input_channels'])

# [0,0] Loss — train AND val (key for finding inflection point)
axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
if history.get('val_loss'):
    axes[0, 0].plot(history['val_loss'], label='Val Loss', linewidth=2, linestyle='--', color='red')
    # Mark best val_loss epoch
    best_epoch = history['val_loss'].index(min(history['val_loss']))
    axes[0, 0].axvline(x=best_epoch, color='green', linestyle=':', alpha=0.7,
                       label=f'Best val_loss (epoch {best_epoch+1})')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Train vs Val Loss\n(inflection = your t_max)', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# [0,1] Accuracy — train, val, per-stream
axes[0, 1].plot([a*100 for a in history['train_accuracy']], label='Train', linewidth=2, color='green')
if history.get('val_accuracy'):
    axes[0, 1].plot([a*100 for a in history['val_accuracy']], label='Val', linewidth=2, color='darkgreen', linestyle='--')
for i in range(n_streams):
    key = f'stream_{i}_train_acc'
    if key in history and history[key]:
        axes[0, 1].plot([a*100 for a in history[key]],
            label=f'{STREAM_LABELS[i]} Train', linewidth=1, alpha=0.6, linestyle='--',
            color=stream_train_colors[i % len(stream_train_colors)])
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].set_title('Training Accuracy', fontweight='bold')
axes[0, 1].legend(fontsize=9, loc='lower right')
axes[0, 1].grid(True, alpha=0.3)

# [0,2] Learning Rate (Plateau steps visible here)
if history.get('learning_rates'):
    sampled = history['learning_rates'][::max(1, len(history['learning_rates'])//100)]
    axes[0, 2].plot(sampled, linewidth=2, color='green', label='Base LR')
for i in range(n_streams):
    key = f'stream_{i}_lr'
    if key in history and history[key]:
        axes[0, 2].plot(history[key], linewidth=1, alpha=0.7, linestyle='--',
            color=lr_colors[i % len(lr_colors)], label=f'{STREAM_LABELS[i]} LR')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Learning Rate')
axes[0, 2].set_title('LR Schedule (Plateau Steps)', fontweight='bold')
axes[0, 2].legend(fontsize=9)
axes[0, 2].grid(True, alpha=0.3)

# [1,0] Gradient norms
if 'gradient_norms' in history and history['gradient_norms']:
    grad_epochs = range(len(history['gradient_norms']))
    for i in range(n_streams):
        norms = [d.get(f'stream_{i}', {}).get('mean', 0) for d in history['gradient_norms']]
        axes[1, 0].plot(grad_epochs, norms, label=f'{STREAM_LABELS[i]}',
            color=stream_val_colors[i % len(stream_val_colors)], linewidth=1.5)
    shared_norms = [d.get('shared', {}).get('mean', 0) for d in history['gradient_norms']]
    axes[1, 0].plot(grad_epochs, shared_norms, label='Shared', color='gray', linewidth=1.5, linestyle='--')
    axes[1, 0].set_yscale('log')
    axes[1, 0].legend(fontsize=9)
else:
    axes[1, 0].text(0.5, 0.5, 'No gradient data', ha='center', va='center', transform=axes[1, 0].transAxes)
axes[1, 0].set_title('Gradient Norms', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# [1,1] Per-stream contribution
import math
contrib_key = 'stream_0_train_acc'
if contrib_key in history and history[contrib_key]:
    baseline_vals = history['train_accuracy']
    for i in range(n_streams):
        other = (i + 1) % n_streams if n_streams == 2 else i
        other_vals = history[f'stream_{other}_train_acc']
        contrib = []
        epochs_eval = []
        for e, (other_acc, base) in enumerate(zip(other_vals, baseline_vals)):
            if not math.isnan(other_acc):
                contrib.append((base - other_acc) * 100)
                epochs_eval.append(e)
        axes[1, 1].plot(epochs_eval, contrib,
            label=f'{STREAM_LABELS[i]}', color=stream_val_colors[i % len(stream_val_colors)],
            linewidth=1.5, marker='o', markersize=3)
    axes[1, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[1, 1].legend(fontsize=9)
else:
    axes[1, 1].text(0.5, 0.5, 'No stream data', ha='center', va='center', transform=axes[1, 1].transAxes)
axes[1, 1].set_title('Per-Stream Contribution', fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Contribution (pp)')
axes[1, 1].grid(True, alpha=0.3)

# [1,2] Gradient health summary
if 'gradient_health' in history and history['gradient_health']:
    axes[1, 2].axis('off')
    health_text = "Gradient Health Summary:\n\n"
    status_counts = {}
    for h in history['gradient_health']:
        status = h.get('status', 'unknown') if isinstance(h, dict) else str(h)
        status_counts[status] = status_counts.get(status, 0) + 1
    for status, count in sorted(status_counts.items(), key=lambda x: -x[1]):
        health_text += f"  {status}: {count} epochs\n"
    axes[1, 2].text(0.1, 0.9, health_text, transform=axes[1, 2].transAxes,
        fontsize=10, verticalalignment='top', fontfamily='monospace')
else:
    axes[1, 2].text(0.5, 0.5, 'No gradient health data', ha='center', va='center', transform=axes[1, 2].transAxes)
axes[1, 2].set_title('Gradient Health', fontweight='bold')

plt.tight_layout()
plt.savefig(f"{checkpoint_dir}/training_diagnostics.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved: {checkpoint_dir}/training_diagnostics.png")


## 15. Integration Weight Evolution During Training

How did the learned integration priorities change over training? Did the model start RGB-heavy and shift toward Depth?

In [ ]:
# Integration weight evolution visualization
evo_viz = IntegrationWeightEvolutionVisualizer(stream_labels=STREAM_LABELS)

# Plot norm evolution from training history
if 'integration_weight_norms' in history:
    evo_viz.plot_norm_evolution(history, save_path=f"{checkpoint_dir}/integration_weight_evolution.png")
    print(f"Integration weight norm evolution saved.")
else:
    print("No integration weight norm data found in history.")

# Plot full weight snapshots if saved
snapshot_dir = TRAIN_CONFIG.get('integration_snapshot_path')
if snapshot_dir and os.path.isdir(snapshot_dir) and os.listdir(snapshot_dir):
    evo_viz.plot_snapshot_heatmaps(snapshot_dir, save_path=f"{checkpoint_dir}/integration_weight_snapshots.png")
    print(f"Integration weight snapshot heatmaps saved.")
else:
    print("No integration weight snapshots found.")

## 16. Save Results & Model

In [ ]:
import json
import torch

print("=" * 60)
print("SAVING RESULTS")
print("=" * 60)

# Save training history as JSON
history_path = f"{checkpoint_dir}/training_history.json"
with open(history_path, 'w') as f:
    # Build pathway analysis dict with all returned data
    pa_json = {
        'accuracy': {k: float(v) for k, v in pathway_analysis['accuracy'].items()},
        'loss': {k: float(v) for k, v in pathway_analysis['loss'].items()},
        'feature_norms': {k: float(v) for k, v in pathway_analysis['feature_norms'].items()},
        'samples_analyzed': pathway_analysis['samples_analyzed'],
    }

    json_history = {
        'train_loss': [float(x) for x in history['train_loss']],
        'train_accuracy': [float(x) for x in history['train_accuracy']],
        'learning_rates': [float(x) for x in history['learning_rates']],
        'model_config': MODEL_CONFIG,
        'dataset_config': DATASET_CONFIG,
        'augmentation_config': AUGMENTATION_CONFIG.to_dict(),
        'stream_specific_config': STREAM_SPECIFIC_CONFIG,
        'scheduler_config': SCHEDULER_CONFIG,
        'training_config': {k: str(v) if not isinstance(v, (int, float, bool, type(None))) else v for k, v in TRAIN_CONFIG.items()},
        'test_results': {
            'loss': float(results['loss']),
            'accuracy': float(results['accuracy'])
        },
        'pathway_analysis': pa_json,
    }

    json.dump(json_history, f, indent=2)

print(f"Training history saved: {history_path}")

# Save final model
final_model_path = f"{checkpoint_dir}/final_model.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': model.optimizer.state_dict(),
    'scheduler_state_dict': model.scheduler.state_dict() if model.scheduler else None,
    'config': MODEL_CONFIG,
    'history': history,
    'test_accuracy': results['accuracy']
}, final_model_path)

print(f"Final model saved: {final_model_path}")

# List saved files
print(f"\nAll results saved to: {checkpoint_dir}")
!ls -lh {checkpoint_dir}

print("\n" + "=" * 60)

## 17. Internal CNN Visualization Suite

Everything below runs on the **trained model** with the **test set**. Each cell is independent — run whichever analyses interest you.

In [ ]:
# --- 17a. Feature Map Visualization ---
# "What does the CNN see at each layer?"
# Three modes: full model, single-stream isolated, ablation
# Compare layer1 (early/texture) vs layer4 (late/semantic)

fm_viz = FeatureMapVisualizer(model, stream_labels=STREAM_LABELS)

# Get a single test sample
test_iter = iter(test_loader)
sample_batch = next(test_iter)
*stream_batches, labels = sample_batch
# Take first sample
stream_inputs = [s[0:1].to(model.device) for s in stream_batches]

print(f"Sample label: {labels[0].item()} ({class_names[labels[0].item()] if 'class_names' in dir() else '?'})")

for layer in ['layer1', 'layer4']:
    print(f"\n{'='*60}")
    print(f"  {layer.upper()} FEATURE MAPS")
    print(f"{'='*60}")

    # Mode 1: Full model view (all streams + integrated)
    print(f"\n--- Full Model View ({layer}) ---")
    fm_viz.visualize(stream_inputs, layer=layer, top_k=8,
                     save_path=f"{checkpoint_dir}/featuremaps_full_{layer}.png")

    # Mode 2: Per-stream isolated views
    for i, label in STREAM_LABELS.items():
        print(f"\n--- {label} Stream Isolated View ({layer}) ---")
        fm_viz.visualize(stream_inputs, layer=layer, mode='stream', stream_idx=i, top_k=8,
                         save_path=f"{checkpoint_dir}/featuremaps_{label.lower()}_{layer}.png")

    # Mode 3: Ablation (what happens when we remove a stream?)
    for i, label in STREAM_LABELS.items():
        print(f"\n--- Ablation: {label} Blanked ({layer}) ---")
        fm_viz.visualize(stream_inputs, layer=layer, mode='ablation', stream_idx=i, top_k=8,
                         save_path=f"{checkpoint_dir}/featuremaps_ablation_{label.lower()}_{layer}.png")

# Batch-averaged feature maps at both layers
for layer in ['layer1', 'layer4']:
    print(f"\n--- Batch-Averaged Feature Maps ({layer}, 32 samples) ---")
    fm_viz.visualize_batch(test_loader, layer=layer, n=32, top_k=8,
                           save_path=f"{checkpoint_dir}/featuremaps_batch_avg_{layer}.png")

print("\nFeature map visualizations complete!")

In [ ]:
# --- 17b. Stream Contribution Decomposition ---
# THE unique LINet3 visualization: how much does each stream contribute
# to each neuron's activation in the integrated pathway?

contrib_viz = StreamContributionVisualizer(model, stream_labels=STREAM_LABELS)

# Single image contribution at layer4
print("--- Stream Contributions (layer4, single sample) ---")
contrib_viz.visualize(stream_inputs, layer='layer4',
                      save_path=f"{checkpoint_dir}/contributions_layer4.png")

# Batch-averaged contributions (more representative)
print("\n--- Batch-Averaged Contributions (layer4, 32 samples) ---")
contrib_viz.visualize_batch(test_loader, layer='layer4', n=32,
                            save_path=f"{checkpoint_dir}/contributions_batch_layer4.png")

# Multi-layer comparison
for layer in ['layer1', 'layer2', 'layer3', 'layer4']:
    print(f"\n--- Contributions at {layer} ---")
    contrib_viz.visualize(stream_inputs, layer=layer,
                          save_path=f"{checkpoint_dir}/contributions_{layer}.png")

print("\nStream contribution decomposition complete!")

In [ ]:
# --- 17c. Stream-Decomposed Grad-CAM ---
# Where does each stream focus its attention?

gradcam = StreamGradCAM(model, stream_labels=STREAM_LABELS)

# Integrated Grad-CAM (standard: where does the full model look?)
print("--- Integrated Grad-CAM (layer4) ---")
gradcam.visualize(stream_inputs, layer='layer4', mode='integrated',
                  save_path=f"{checkpoint_dir}/gradcam_integrated_layer4.png")

# Per-stream isolated Grad-CAM (where does each stream look independently?)
for i, label in STREAM_LABELS.items():
    print(f"\n--- {label} Stream Grad-CAM (layer4) ---")
    gradcam.visualize(stream_inputs, layer='layer4', mode='stream', stream_idx=i,
                      save_path=f"{checkpoint_dir}/gradcam_{label.lower()}_layer4.png")

# Decomposed mode: contribution maps weighted by Grad-CAM importance
print("\n--- Decomposed Grad-CAM (layer4) ---")
gradcam.visualize(stream_inputs, layer='layer4', mode='decomposed',
                  save_path=f"{checkpoint_dir}/gradcam_decomposed_layer4.png")

# Multi-layer Grad-CAM (early=texture, late=semantics)
for layer in ['layer2', 'layer3', 'layer4']:
    print(f"\n--- Integrated Grad-CAM at {layer} ---")
    gradcam.visualize(stream_inputs, layer=layer, mode='integrated',
                      save_path=f"{checkpoint_dir}/gradcam_integrated_{layer}.png")

print("\nGrad-CAM visualizations complete!")

In [ ]:
# --- 17d. Integration Weight Visualization ---
# Visualize the learned integration_from_streams weights per layer

iw_viz = IntegrationWeightVisualizer(model, stream_labels=STREAM_LABELS)

# Weight heatmaps per layer and stream
print('--- Integration Weights (Heatmaps) ---')
iw_viz.visualize_weights(save_path=f'{checkpoint_dir}/integration_weights.png')

# Cross-stream comparison (relative weight magnitudes per layer)
print('\n--- Cross-Stream Weight Comparison ---')
iw_viz.visualize_cross_stream(save_path=f'{checkpoint_dir}/integration_cross_stream.png')

# Effective rank via SVD (how low-dimensional is the integration?)
print('\n--- Effective Rank (SVD) ---')
ranks = iw_viz.compute_effective_rank()
for layer, r in ranks.items():
    print(f'  {layer}: {[f"{x:.1f}" for x in r]}')

print('\nIntegration weight visualization complete!')

In [ ]:
# --- 17e. Stream Redundancy Analysis ---
# Are RGB and Depth learning the same features? Or complementary ones?
# Uses centered cosine similarity between stream feature maps at each layer.

redundancy = StreamRedundancyAnalyzer(model, stream_labels=STREAM_LABELS)

print('--- Stream Redundancy (Centered Cosine Similarity) ---')
sim_results = redundancy.analyze(
    test_loader,
    n=128,  # Average over 128 samples
    save_path=f'{checkpoint_dir}/stream_redundancy.png'
)

# Print similarity matrices
for layer_name, sim_matrix in sim_results.items():
    print(f'\n{layer_name}:')
    for i in range(sim_matrix.shape[0]):
        row = '  '.join(f'{sim_matrix[i,j]:.3f}' for j in range(sim_matrix.shape[1]))
        print(f'  {STREAM_LABELS.get(i, f"S{i}")}: {row}')

print('\nStream redundancy analysis complete!')

In [ ]:
# --- 17f. Per-Class Stream Dominance ---
# Which scenes rely on RGB vs Depth?
# "Depth matters more for bathrooms, RGB dominates corridors"

# Build class name mapping
class_name_map = {i: name for i, name in enumerate(class_names)} if 'class_names' in dir() else None

dominance = PerClassDominanceAnalyzer(model, stream_labels=STREAM_LABELS)

print('--- Per-Class Stream Dominance (layer4) ---')
class_dominance = dominance.analyze(
    test_loader,
    layer='layer4',
    class_names=class_name_map,
    save_path=f'{checkpoint_dir}/per_class_dominance.png'
)

# Print per-class ratios
print('\nPer-class stream contribution ratios:')
for cls_idx, ratios in sorted(class_dominance.items()):
    name = class_name_map[cls_idx] if class_name_map else f'Class {cls_idx}'
    ratio_str = ', '.join(f'{STREAM_LABELS.get(i, f"S{i}")}: {r:.2%}' for i, r in enumerate(ratios))
    print(f'  {name}: {ratio_str}')

print('\nPer-class dominance analysis complete!')

In [ ]:
# --- 17g. Misclassification Analysis + Sample Comparison ---
# Find misclassified samples and compare with correctly classified ones

print('--- Finding Misclassified Samples ---')
misclassified = find_misclassified(model, test_loader, n=10)

print(f'Found {len(misclassified)} misclassified samples:')
for i, mc in enumerate(misclassified[:5]):
    true_name = class_names[mc['true_label']] if 'class_names' in dir() else str(mc['true_label'])
    pred_name = class_names[mc['predicted_label']] if 'class_names' in dir() else str(mc['predicted_label'])
    print(f'  [{i}] True: {true_name}, Predicted: {pred_name}, Confidence: {mc["confidence"]:.2%}')

# Grad-CAM on first misclassified sample
if misclassified:
    mc_sample = misclassified[0]
    mc_inputs = [s.to(model.device) for s in mc_sample['stream_inputs']]
    true_name = class_names[mc_sample['true_label']] if 'class_names' in dir() else str(mc_sample['true_label'])
    pred_name = class_names[mc_sample['predicted_label']] if 'class_names' in dir() else str(mc_sample['predicted_label'])
    print(f'\n--- Grad-CAM on Misclassified: True={true_name}, Pred={pred_name} ---')
    gradcam.visualize(mc_inputs, layer='layer4', mode='decomposed',
                      save_path=f'{checkpoint_dir}/gradcam_misclassified_0.png')

# Compare correct vs misclassified from same class
if misclassified:
    target_class = misclassified[0]['true_label']
    print(f'\n--- Finding correctly classified sample from class {class_names[target_class] if "class_names" in dir() else target_class} ---')

    # Find a correctly classified sample from the same class
    correct_sample = None
    model.eval()
    with torch.no_grad():
        for batch_data in test_loader:
            *stream_batches, targets = batch_data
            stream_batches_dev = [s.to(model.device) for s in stream_batches]
            targets_dev = targets.to(model.device)
            logits = model(stream_batches_dev)
            preds = logits.argmax(dim=1)
            # Find correctly classified samples of the target class
            mask = (targets_dev == target_class) & (preds == target_class)
            if mask.any():
                idx = mask.nonzero(as_tuple=True)[0][0].item()
                correct_sample = {
                    'stream_inputs': [s[idx:idx+1].cpu() for s in stream_batches],
                    'true_label': target_class,
                    'predicted_label': target_class,
                    'confidence': torch.softmax(logits[idx], dim=0)[target_class].item(),
                }
                break

    if correct_sample is not None:
        print(f'  Found correct sample (confidence: {correct_sample["confidence"]:.2%})')
        print('\n--- Correct vs Misclassified Comparison ---')
        compare_samples(
            model,
            correct_sample=correct_sample,
            misclassified_sample=misclassified[0],
            layer='layer4',
            stream_labels=STREAM_LABELS,
            save_path=f'{checkpoint_dir}/compare_samples.png'
        )
    else:
        print('  No correctly classified sample found for this class.')

print('\nMisclassification analysis complete!')

In [ ]:
# --- 17h. Train vs Test Activation Divergence ---
# Does the model see different activation distributions on train vs test?
# Uses MMD (Maximum Mean Discrepancy) per layer.

div_analyzer = ActivationDivergenceAnalyzer(model)

print('--- Train vs Test Activation Divergence ---')
divergence = div_analyzer.analyze(
    train_loader,
    test_loader,
    n=128,
    save_path=f'{checkpoint_dir}/activation_divergence.png'
)

for layer_name, metrics in divergence.items():
    print(f'  {layer_name}: MMD={metrics["mmd"]:.4f}')

print('\nActivation divergence analysis complete!')

In [ ]:
# --- 17i. BN Stats Reset Experiment (Oracle Diagnostic) ---
# WARNING: This is a DIAGNOSTIC tool, not a deployable fix.
# It recomputes BN running stats on test data (oracle) to check if
# BN statistics drift causes the generalization gap.

import copy

# Save original accuracy
original_test_results = model.evaluate(test_loader)
original_acc = original_test_results['accuracy']
print(f'Original test accuracy: {original_acc*100:.2f}%')

# Control: recompute BN stats on TRAIN set (should be ~same)
print('\n--- Control: Recompute BN stats on TRAIN set ---')
model_control = copy.deepcopy(model)
reset_bn_stats(model_control, train_loader)
control_results = model_control.evaluate(test_loader)
control_acc = control_results['accuracy']
print(f'After train BN reset: {control_acc*100:.2f}% (delta: {(control_acc-original_acc)*100:+.2f}%)')

# Oracle: recompute BN stats on TEST set
print('\n--- Oracle: Recompute BN stats on TEST set ---')
model_oracle = copy.deepcopy(model)
reset_bn_stats(model_oracle, test_loader)
oracle_results = model_oracle.evaluate(test_loader)
oracle_acc = oracle_results['accuracy']
print(f'After test BN reset (oracle): {oracle_acc*100:.2f}% (delta: {(oracle_acc-original_acc)*100:+.2f}%)')

# Interpretation
print('\n--- Interpretation ---')
oracle_delta = (oracle_acc - original_acc) * 100
if abs(oracle_delta) > 2:
    print(f'BN stats drift accounts for ~{oracle_delta:+.1f}% of the gap.')
    print('Consider: test-time BN adaptation, larger batch size, or more training data.')
else:
    print(f'BN stats drift is minimal ({oracle_delta:+.1f}%). Gap is likely from other sources.')

del model_control, model_oracle  # Free memory
print('\nBN reset experiment complete!')

## 18. Summary

All training diagnostics and visualization analyses are saved to the checkpoint directory on Google Drive.

**Saved models:**
- `best_model.pt` - Best model checkpoint (by training loss, since no val set)
- `final_model.pt` - Final model with full state dict, optimizer, scheduler, history

**Saved data:**
- `training_history.json` - Full training history, configs, test results, pathway analysis
- `integration_snapshots/` - Periodic full integration weight snapshots (every N epochs)

**Saved visualizations:**
- `training_diagnostics.png` - 2x3 grid: loss, accuracy, LR, gradient norms, stream losses, gradient health
- `integration_weight_evolution.png` - Per-stream integration weight norms over training epochs
- `integration_weight_snapshots.png` - Detailed weight heatmaps at snapshot epochs
- `featuremaps_*.png` - What the CNN sees (full model, per-stream isolated, ablation, batch-averaged)
- `contributions_*.png` - Per-stream contribution magnitudes at each layer
- `gradcam_*.png` - Spatial attention maps (integrated, per-stream, decomposed, multi-layer)
- `gradcam_misclassified_0.png` - Decomposed Grad-CAM on a misclassified sample
- `integration_weights.png` - Learned fusion weight heatmaps per layer
- `integration_cross_stream.png` - Cross-stream weight magnitude comparison
- `stream_redundancy.png` - Centered cosine similarity between stream features per layer
- `per_class_dominance.png` - Which scenes rely on RGB vs Depth
- `activation_divergence.png` - Train vs test activation distribution shift (MMD) per layer
- `compare_samples.png` - Correct vs misclassified side-by-side (Grad-CAM + contributions)